# Football: exploration and first YOLO baseline
## Goal
Run one complete development workflow: audit images and annotations, visualize ground truth, infer with YOLO26n, evaluate and compare 640 vs 1280.

**Status:** Prepared notebook; no real-sample results executed or claimed in this file. Use your seven original images and their source annotations. Restart the kernel and run all cells in order.

**Scope:** Development sample only, not an independent test set. No training, mAP, team assignment or possession analysis.

**Method:** Dataset `player` and `referee` map to `person`; dataset `ball` maps to `ball`. Model `person` and `sports ball` map accordingly. Matching is per image and class, greedy by descending confidence, one prediction per GT, IoU >= configured threshold. Unmatched predictions are FP; unmatched annotations are FN. Undefined ratios are `None`, not zero.

**Important:** Unannotated real people outside the pitch count as FP under this reference set. Inspect such cases before interpreting precision. Annotation errors are not silently corrected. Empty labels are allowed but require visual review. Tiny boxes raise warnings.

## 1. Setup
Select the existing project `.venv` as notebook kernel. Dependencies: installed CPU requirements plus `requirements-notebook.txt`. Paths in `configs/baseline.yaml` are relative to the repository root. The source dataset YAML supplies only class names, not training paths.

In [ ]:
from pathlib import Path
import sys, json, csv, platform, subprocess, importlib.metadata, html
from dataclasses import asdict
from collections import Counter
from datetime import datetime, timezone
import yaml
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'configs' / 'baseline.yaml').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook inside the project; configs/baseline.yaml not found')
sys.path.insert(0, str(ROOT))
from sports_analyzer.annotations.yolo import load_names
from sports_analyzer.baseline_data import load_sample, sha256, show_grid
from sports_analyzer.inference import Detector
from sports_analyzer.evaluation import remap, GT_MAPPING, PRED_MAPPING, evaluate_image, aggregate

plt.rcParams.update({'figure.dpi': 100, 'font.size': 10})
def table(rows):
    if not rows:
        print('No rows')
        return
    keys = list(rows[0])
    head = ''.join('<th>'+html.escape(str(k))+'</th>' for k in keys)
    body = ''.join('<tr>'+''.join('<td>'+html.escape(str(r.get(k, '')))+'</td>' for k in keys)+'</tr>' for r in rows)
    display(HTML('<table><thead><tr>'+head+'</tr></thead><tbody>'+body+'</tbody></table>'))

config_path = ROOT / 'configs' / 'baseline.yaml'
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
required = {'images','labels','dataset_yaml','model','output','image_sizes','confidence','match_iou'}
if not isinstance(config, dict) or set(config) != required:
    raise ValueError(f'Config must contain exactly {sorted(required)}')
if not isinstance(config['image_sizes'], list) or not config['image_sizes'] or any(type(v) is not int or v <= 0 for v in config['image_sizes']):
    raise ValueError('image_sizes must contain positive integers')
if len(set(config['image_sizes'])) != len(config['image_sizes']):
    raise ValueError('Duplicate image sizes')
if not 0 <= config['confidence'] <= 1 or not 0 < config['match_iou'] <= 1:
    raise ValueError('Invalid confidence or matching IoU')
print('Project:', ROOT)
print('Python:', sys.executable)
display(config)

## 2. Audit source images and labels
Stop on missing/invalid labels or undecodable images. Byte-identical duplicates are reported; originals remain unchanged. EXIF-rotated images require an explicit coordinate policy and are rejected.

The earlier `data_audit.py` remains useful as a standalone image audit. This paired loader additionally verifies labels and hashes both sources.

In [ ]:
dataset_yaml = ROOT / config['dataset_yaml']
names = load_names(dataset_yaml)
if set(names.values()) != set(GT_MAPPING):
    raise ValueError(f'Unexpected dataset classes: {names}; review mappings before evaluation')
records = load_sample(ROOT / config['images'], ROOT / config['labels'], names)
inventory = [{'filename': r['path'].name, 'width': r['width'], 'height': r['height'],
              'mode': r['mode'], 'objects': len(r['ground_truth']),
              'image_sha256': r['image_sha256'], 'label_sha256': r['label_sha256']} for r in records]
table(inventory)
hashes = Counter(r['image_sha256'] for r in records)
print('Byte-identical duplicate groups:', sum(count > 1 for count in hashes.values()))
print('Images with empty labels:', [r['path'].name for r in records if not r['ground_truth']])

## 3. Explore the sample
Inspect originals and ground truth separately. Check tiny boxes, occlusions and missing labels. The charts describe this sample only; they do not establish dataset representativeness.

In [ ]:
fig = show_grid(records, title='Original development images')
plt.show()
gt_original = {r['path'].name: r['ground_truth'] for r in records}
fig = show_grid(records, gt_original, 'Source annotations: player / referee / ball')
plt.show()

In [ ]:
counts = Counter(b.label for r in records for b in r['ground_truth'])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
classes = ['player', 'referee', 'ball']
colors = ['#0072B2', '#CC79A7', '#D55E00']
axes[0].bar(classes, [counts[c] for c in classes], color=colors)
axes[0].set(title='Annotated objects in this sample', ylabel='Number of boxes')
for label, color in zip(classes, colors):
    boxes = [b for r in records for b in r['ground_truth'] if b.label == label]
    axes[1].scatter([b.xyxy[2]-b.xyxy[0] for b in boxes],
                    [b.xyxy[3]-b.xyxy[1] for b in boxes], label=label, color=color, alpha=.7)
axes[1].set(title='Original-image box dimensions', xlabel='Width (pixels)', ylabel='Height (pixels)')
axes[1].legend()
fig.tight_layout()
plt.show()

## 4. Create a reproducible run
A new output directory records configuration, package versions, Git status, input hashes and local checkpoint hash. A manifest remains `incomplete` if later cells fail. Source images and model weights are not copied or uploaded.

Timing is exploratory: one warm-up per resolution, then CPU predict-call duration and Ultralytics component times. Model loading, export and plotting are excluded. This is not a controlled benchmark.

In [ ]:
model_path = ROOT / config['model']
if not model_path.is_file():
    raise FileNotFoundError(f'Place your previously downloaded yolo26n.pt here: {model_path}')
run_dir = ROOT / config['output'] / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
run_dir.mkdir(parents=True, exist_ok=False)

def git_info(args):
    result = subprocess.run(['git', *args], cwd=ROOT, capture_output=True, text=True)
    return result.stdout.strip() if result.returncode == 0 else {'error': result.stderr.strip()}

manifest = {'status': 'incomplete', 'scope': 'development sample; not held-out evaluation',
            'config': config, 'python': sys.version, 'platform': platform.platform(),
            'git_commit': git_info(['rev-parse','HEAD']), 'git_status': git_info(['status','--porcelain']),
            'packages': {p: importlib.metadata.version(p) for p in ['torch','torchvision','ultralytics','Pillow','PyYAML','matplotlib']},
            'model_sha256': sha256(model_path), 'dataset_yaml_sha256': sha256(dataset_yaml),
            'source_hashes': {str(p.relative_to(ROOT)):sha256(p) for p in sorted((ROOT/'sports_analyzer').rglob('*.py'))},
            'inventory': inventory, 'gt_mapping': GT_MAPPING, 'prediction_mapping': PRED_MAPPING,
            'matching': 'per-image/class, descending confidence, best unmatched IoU, inclusive threshold'}

def save_json(name, value):
    (run_dir/name).write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')

save_json('manifest.json', manifest)
fig = show_grid(records, gt_original, 'Ground truth')
fig.savefig(run_dir/'ground_truth.png', bbox_inches='tight')
plt.close(fig)
print('Run directory:', run_dir)

## 5. Run inference and evaluate
All model classes are retained in `predictions.json`. Evaluation includes only mapped person/ball predictions. No confidence tuning against an independent test set occurs here. Each FP/FN is retained with coordinates for inspection.

In [ ]:
detector = Detector(model_path)
if not {'person','sports ball'}.issubset(set(detector.model.names.values())):
    raise ValueError('Expected COCO person/sports ball classes; update mapping for other weights')
all_summaries, all_details, all_events, all_predictions, timings = [], [], [], {}, []
for size in config['image_sizes']:
    detector.predict(records[0]['path'], size, config['confidence'])  # excluded warm-up
    predictions_by_name = {}
    per_image_rows = []
    for record in records:
        name = record['path'].name
        predictions, timing = detector.predict(record['path'], size, config['confidence'])
        predictions_by_name[name] = remap(predictions, PRED_MAPPING, strict=False)
        ground_truth = remap(record['ground_truth'], GT_MAPPING, strict=True)
        rows, events = evaluate_image(ground_truth, predictions_by_name[name], config['match_iou'])
        per_image_rows.extend(rows)
        all_details.extend({'image': name, 'imgsz': size, **row} for row in rows)
        all_events.extend({'image': name, 'imgsz': size, **event} for event in events)
        timings.append({'image': name, 'imgsz': size, **timing})
        all_predictions[f'{size}/{name}'] = [asdict(b) for b in predictions]
    all_summaries.extend({'imgsz': size, **row} for row in aggregate(per_image_rows))
    fig = show_grid(records, predictions_by_name, f'Predictions: imgsz={size}, conf={config["confidence"]}')
    fig.savefig(run_dir/f'predictions_{size}.png', bbox_inches='tight')
    plt.show()
    save_json('predictions.json', all_predictions)
    save_json('events.json', all_events)

table(all_summaries)

## 6. Results and error inspection
Precision = TP/(TP+FP); recall = TP/(TP+FN). Counts are summed across images before calculating ratios. These are fixed-threshold metrics, not AP/mAP. Matching IoU is distinct from an inference NMS setting.

Do not assume more detections means improvement. Review false positives against the original images: some may be real but unannotated people.

In [ ]:
table(all_details)
latency_rows = []
for size in config['image_sizes']:
    selected = [r for r in timings if r['imgsz'] == size]
    latency_rows.append({'imgsz':size, 'images':len(selected),
        'mean_predict_call_ms': round(sum(r['predict_call_ms'] for r in selected)/len(selected),2),
        'mean_inference_ms': round(sum(r['inference'] for r in selected)/len(selected),2)})
table(latency_rows)
errors = [r for r in all_events if r['status'] != 'TP']
print('Unmatched predictions / annotations:', len(errors))
table(errors[:30])
print('Full event list will be saved as events.json')

In [ ]:
for filename, rows in [('metrics.csv',all_summaries), ('per_image_metrics.csv',all_details),
                       ('timings.csv',timings), ('inventory.csv',inventory)]:
    with (run_dir/filename).open('w',encoding='utf-8',newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
save_json('predictions.json', all_predictions)
save_json('events.json', all_events)
manifest['status'] = 'complete'
save_json('manifest.json', manifest)
print('Completed:', run_dir)

## 7. Interpretation — complete after running
- Which resolution improved ball/person precision or recall on this sample?
- What additional CPU time did it require?
- Which apparent FP are genuinely incorrect, and which are annotation-scope mismatches?
- Inspect the unusually narrow player box in `20250710_144746_004987.txt` before trusting its contribution.
- Record decisions and limitations, not only the preferred result.

**Next:** retain this as development evidence, inspect a larger dataset and split by match/clip before model selection and final testing. Add YOLOv8/YOLO12 and W&B through the same pipeline. Use a standard evaluator for mAP.

Sources: [dataset](https://github.com/skaczylo/Football-Image-Dataset), [Ultralytics predict API](https://docs.ultralytics.com/modes/predict/).